In [ ]:
# Colab setup: mount Drive and move into the project folder (edit PROJECT if your folder is elsewhere)
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/Colab Notebooks/CapstoneProject/careloop-no-show-guard"
    os.chdir(PROJECT)
except ImportError:
    pass                                   # not in Colab: stay in the current folder
print(os.getcwd()); print(os.listdir("."))

# 03 – No-show model (Part 3)
Do **not** shuffle or re-sort `appointments_clean.csv` after loading.

## Task 3.1 – features, split, baseline

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

df = pd.read_csv("appointments_clean.csv")
X = df[["age","distance_km","fee_inr","lead_days","prior_no_shows","reminder_sent","booking_channel","visit_type"]].copy()
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
X = pd.get_dummies(X, columns=["booking_channel","visit_type"]).astype(float)
y = df["no_show"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("X columns:", X.shape[1], list(X.columns))
print("train:", len(X_train), "test:", len(X_test), "| test no-shows:", y_test.sum())
baseline = (y_test == 0).mean()
print("baseline accuracy (always Attended):", round(baseline, 4), f"({(y_test==0).sum()} of {len(y_test)})")

## Task 3.2 – logistic regression

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
lr = LogisticRegression(max_iter=1000).fit(X_train_s, y_train)
pred = lr.predict(X_test_s)
prob = lr.predict_proba(X_test_s)[:, 1]
print("accuracy", round(accuracy_score(y_test, pred), 3))
print(confusion_matrix(y_test, pred))
print("precision", round(precision_score(y_test, pred), 3), "recall", round(recall_score(y_test, pred), 3),
      "F1", round(f1_score(y_test, pred), 3), "AUC", round(roc_auc_score(y_test, prob), 3))
for name, c in zip(X.columns, lr.coef_[0]):
    print(f"{name:28s}{c:+.3f}")
i = list(X.columns).index("reminder_sent")
coef = lr.coef_[0][i]
print("reminder_sent: scaler.scale_ =", round(scaler.scale_[i], 3))
print("np.exp(coef) (per 1 SD, scaled)      =", round(np.exp(coef), 3))
print("np.exp(coef/scale) (real No->Yes)    =", round(np.exp(coef / scaler.scale_[i]), 3))

**Interpretation (write in your own words):** the two numbers differ because the first is the odds ratio for a one-standard-deviation move of the *scaled* reminder variable (SD is only about 0.49), while the second rescales the coefficient to the real 0→1 change. Conclusion as an association only: in this model, holding the other features constant, bookings with a reminder have about X% lower odds of a no-show; the data are observational, so this does not prove reminders cause fewer no-shows.

## Task 3.3 – decision trees

In [ ]:
for label, t in [("full depth", DecisionTreeClassifier(random_state=42)),
                 ("max_depth=3", DecisionTreeClassifier(max_depth=3, random_state=42))]:
    t.fit(X_train, y_train)
    print(f"{label}: depth={t.get_depth()} train={t.score(X_train, y_train):.3f} test={t.score(X_test, y_test):.3f}")

**Gap statement (write in your own words):** the full tree scores 1.00 on training but falls below the baseline on test, so it memorised noise; the depth-3 tree has a small gap and generalises.

## Task 3.4 – random forest and comparison table (reporting only)

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
rf_pred, rf_prob = rf.predict(X_test), rf.predict_proba(X_test)[:, 1]
print("RF train acc", round(rf.score(X_train, y_train), 3), "test acc", round(accuracy_score(y_test, rf_pred), 3),
      "recall", round(recall_score(y_test, rf_pred), 3), "AUC", round(roc_auc_score(y_test, rf_prob), 3))
print(pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(3))

tree3 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
t3_pred, t3_prob = tree3.predict(X_test), tree3.predict_proba(X_test)[:, 1]
table = pd.DataFrame({
    "model": ["Logistic regression", "Decision tree (depth 3)", "Random forest"],
    "accuracy": [accuracy_score(y_test, pred), accuracy_score(y_test, t3_pred), accuracy_score(y_test, rf_pred)],
    "recall": [recall_score(y_test, pred), recall_score(y_test, t3_pred), recall_score(y_test, rf_pred)],
    "roc_auc": [roc_auc_score(y_test, prob), roc_auc_score(y_test, t3_prob), roc_auc_score(y_test, rf_prob)]}).round(3)
print(table)

**Which model looks best and why (write in your own words, from your table):** compare ROC-AUC and recall across the three rows; note accuracy alone is misleading because the baseline is about 0.75. This table is for reporting only; the formal choice is made in Task 3.5 from training data.

## Task 3.5 – tune, choose model and threshold on training data only, test once

In [ ]:
# (a) grid search
grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    {"n_estimators": [100, 200], "max_depth": [3, 5, 8]}, cv=5, scoring="roc_auc")
grid.fit(X_train, y_train)
print("best_params_:", grid.best_params_, "| best_score_ (CV ROC-AUC):", round(grid.best_score_, 3))

# (b) choose the model by cross-validated training ROC-AUC
lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
candidates = {
    "Logistic regression": lr_pipe,
    "Decision tree (depth 3)": DecisionTreeClassifier(max_depth=3, random_state=42),
    "Random forest (default)": RandomForestClassifier(n_estimators=100, random_state=42),
}
cv_scores = {k: cross_val_score(m, X_train, y_train, cv=5, scoring="roc_auc").mean() for k, m in candidates.items()}
cv_scores["Tuned random forest"] = grid.best_score_
candidates["Tuned random forest"] = grid.best_estimator_
for k, v in cv_scores.items():
    print(f"{k:26s}{v:.3f}")
winner_name = max(cv_scores, key=cv_scores.get)
winner = candidates[winner_name]
print("WINNER:", winner_name)

# (c) choose the threshold from out-of-fold training probabilities
oof = cross_val_predict(winner, X_train, y_train, cv=5, method="predict_proba")[:, 1]
rows = []
for t in [0.5, 0.35]:
    p = (oof >= t).astype(int)
    rows.append({"threshold": t, "precision": precision_score(y_train, p), "recall": recall_score(y_train, p),
                 "F1": f1_score(y_train, p), "flagged": int(p.sum())})
thr_table = pd.DataFrame(rows).round(3)
print(thr_table)
THRESHOLD = float(thr_table.loc[thr_table["F1"].idxmax(), "threshold"])
print("chosen threshold:", THRESHOLD)

**Why recall matters here (own words):** a missed warning costs an empty doctor slot, while an extra call costs about a minute, so catching more true no-shows is worth some extra calls. **Real-world threshold:** a clinic would set it by how many follow-up calls the front desk can make in an evening, weighed against the cost of an empty slot.

In [ ]:
# (d) fit winner on training set, evaluate ONCE on the test set
winner.fit(X_train, y_train)
test_prob = winner.predict_proba(X_test)[:, 1]
test_flag = (test_prob >= THRESHOLD).astype(int)
print(f"{winner_name} @ {THRESHOLD}: precision={precision_score(y_test, test_flag):.3f} "
      f"recall={recall_score(y_test, test_flag):.3f} flagged={test_flag.sum()} AUC={roc_auc_score(y_test, test_prob):.3f}")

# (e) save scores
scores = pd.DataFrame({"appointment_id": df.loc[X_test.index, "appointment_id"].values,
                       "no_show": y_test.values, "risk_prob": test_prob.round(4), "flag": test_flag})
scores.to_csv("test_risk_scores.csv", index=False)
print(scores.shape, "| rows with flag=1:", scores["flag"].sum())